# Supervised Fine-Tuning (SFT)

Fine-tuned **Qwen2.5-1.5B-Instruct** to solve the Travelling Salesman Problem by imitating an OR-Tools teacher, using LoRA + 4-bit quantization.

**Pipeline:** install -> load model + LoRA -> train on `train.jsonl` -> save adapter to Drive.

**Before running:**
1. Change runtime type to **T4 GPU**.
2. Upload `train.jsonl` (from `make_dataset.py`) into `/content/` via the file panel.

> To reproduce the 0.5B baseline instead, change `MODEL` to `Qwen/Qwen2.5-0.5B-Instruct`.

## 1. Installing dependencies

These exact version pins avoid the dependency conflicts in the Colab base image
(`protobuf` must be ≥5.26 for `transformers` but <6 to avoid an OR-Tools clash;
`torchao` is removed because PEFT errors on an old version if present).

In [ ]:
!pip install -q -U "transformers>=4.46" "trl>=0.12,<0.13" "datasets>=3.0,<3.2" \
    peft accelerate bitsandbytes "protobuf>=5.26,<6" "fsspec<=2024.9.0"
!pip uninstall -q -y torchao

## 2. Loading the model in 4-bit with a LoRA adapter

4-bit quantization shrinks memory ~4x so the model fits a 16 GB T4. LoRA freezes the base model and trains only small adapter matrices (<1% of parameters). This is what makes fine-tuning feasible on a free GPU.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # use Qwen2.5-0.5B-Instruct for the small baseline

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
print("model loaded")

## 3. Training (SFT)

Supervised fine-tuning on the (problem, optimal-tour) pairs. The effective batch size is `per_device_train_batch_size × gradient_accumulation_steps = 2 × 8 = 16`;
gradient accumulation + checkpointing keep peak memory within the T4.

In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

data = load_dataset("json", data_files={"train": "/content/train.jsonl"})

cfg = SFTConfig(
    output_dir="sft-qwen-tsp",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=20,
    max_seq_length=2048,
    bf16=False, fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model, args=cfg,
    train_dataset=data["train"],
    peft_config=lora,
    processing_class=tokenizer,
)
trainer.train()

## 4. Saving the LoRA adapter to Google Drive

Colab sessions are wiped on disconnect, so the trained adapter is saved to Drive. The base model is re-downloaded when reloading.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/llm-tsp/sft-adapter-1.5b'
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print('saved to', save_path)